# CrediX Dual-Engine Fraud Detection Model — Training Notebook

**Version:** v2.5.0-realistic-enterprise  
**Author:** CrediX Quantitative Risk & AI Team  
**Framework:** scikit-learn (Multi-Tree IsolationForest + Cost-Sensitive HistGradientBoosting)  
**Target Market:** Egyptian Retail Banking (aligned with CBE guidelines)  

---

## 1. Overview & Architecture

This notebook documents the training, adversarial calibration, and statistical validation of the **CrediX Dual-Engine Fraud Machine Learning Layer**.

### The 5-Layer Hybrid Fraud Engine:
1. **Layer 1:** Deterministic Cross-Document Validation (NID format, salary slip mismatch, OCR quality, I-Score freshness, statement integrity)
2. **Layer 2:** Deep Forensic Signals (Benford's Law Chi-Square distribution on cash flows, round-number uniformity analysis)
3. **Layer 3:** Real-time Entity Graph & Velocity Defense (cross-application collisions on phone, employer, and NID over 48h windows)
4. **Layer 4:** **Dual-Engine Machine Learning (This Notebook):** Multi-tree Isolation Forest (unsupervised) + Cost-sensitive Gradient Boosting (supervised)
5. **Layer 5:** Cost-Sensitive Risk Aggregation & Arabic/English Explainable AI (XAI)

---
## 2. Dataset Specifications & Regulatory Alignment

> ### ⚠️ DATA GOVERNANCE & PRIVACY NOTICE
> - The dataset `data/fraud_training_data_25000.csv` is **100% synthetic** and contains **no real customer PII** (Personally Identifiable Information).
> - It was algorithmically generated using parametric distributions (Beta, Gamma, Poisson) fitted to retail banking observations aligned with CBE regulations.
> - Features reflect realistic friction: freelancers with erratic cash flows, high-net-worth customers with large transfers, and subtle fraudulent manipulations.

### Dataset Summary:
- **Total Records:** 25,000 credit applications
- **Clean Borrowers:** 23,625 (94.5%)
- **Fraudulent Applications:** 1,375 (5.5% fraud prevalence)
  - **Camouflaged Fraud (Subtle):** 825 cases (60% of fraud) — designed to break naive separation
  - **Flagrant Fraud (Gross):** 550 cases (40% of fraud)
- **Input Features:** 12 engineered forensic & behavioral signals

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve
)
import joblib

print('Libraries and dependencies loaded successfully.')

---
## 3. Loading the Dataset

In [ ]:
data_path = os.path.join('..', 'data', 'fraud_training_data_25000.csv')
if not os.path.exists(data_path):
    data_path = 'fraud_training_data_25000.csv'

df = pd.read_csv(data_path, encoding='utf-8-sig')
print(f'Total Applications: {len(df):,}')
print(f'Feature Columns:    {df.shape[1]}')
print('\nPopulation Breakdown by Class:')
print(df['record_type'].value_counts())

---
## 4. The Synthetic Over-Separation Trap (Overfitting Reality Check)

When teams create synthetic fraud data, they often define fraud as having extreme outliers (e.g. mismatch > 60%, OCR < 50%).
Training any model on such data produces **ROC-AUC = 1.0000** and **Recall = 100%**.

**Why this is dangerous:**
- In real retail banking, fraudsters do not submit completely unreadable documents.
- Sophisticated fraudsters manipulate numbers by 15–25% to stay below heuristic alerts.
- Our calibrated dataset introduces **60% camouflaged fraud** with heavy feature overlap, forcing the model to learn authentic non-linear boundaries.

In [ ]:
FEATURE_NAMES = [
    'income_mismatch_ratio', 'annuity_to_balance_ratio', 'balance_volatility_cv',
    'surge_ratio_max_to_avg', 'ocr_quality_mean', 'min_to_avg_balance_ratio',
    'applicant_age_norm', 'employment_tenure_years', 'inflow_regularity_score',
    'iscore_normalized', 'inflow_uniformity_score', 'bureau_facilities_count'
]

print('Mean Feature Values by Sub-population:')
print(df.groupby('record_type')[FEATURE_NAMES].mean().round(3).T.to_string())

---
## 5. Model Training: Dual-Engine ML Architecture

We train two complementary models:
1. **Isolation Forest (Unsupervised):** Detects structural distribution shifts without relying on labels (catches zero-day fraud schemes).
2. **HistGradientBoosting (Supervised):** Trained with **8x asymmetric loss weighting** to prioritize Recall over False Positives.

In [ ]:
X = df[FEATURE_NAMES].values
y = df['is_fraud'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 1. Unsupervised Engine: Isolation Forest
print('[*] Training Isolation Forest (200 trees)...')
iso_model = IsolationForest(
    n_estimators=200,
    contamination=0.055,
    max_samples=256,
    random_state=42,
    n_jobs=-1
)
iso_model.fit(X_train_scaled)
raw_test_scores = iso_model.decision_function(X_test_scaled)
iso_scores = np.clip(0.50 - (raw_test_scores * 1.8), 0.0, 1.0)

# 2. Supervised Engine: Cost-Sensitive Gradient Boosting (8x Penalty)
print('[*] Training HistGradientBoosting (8x Fraud Weight)...')
sample_weights = np.where(y_train == 1, 8.0, 1.0)
gb_model = HistGradientBoostingClassifier(
    max_iter=140,
    learning_rate=0.06,
    max_leaf_nodes=25,
    min_samples_leaf=40,
    l2_regularization=3.0,
    random_state=42
)
gb_model.fit(X_train_scaled, y_train, sample_weight=sample_weights)
gb_probs = gb_model.predict_proba(X_test_scaled)[:, 1]

# 3. Hybrid Fusion: 35% Unsupervised Anomaly + 65% Supervised Probability
hybrid_probs = (0.35 * iso_scores) + (0.65 * gb_probs)
threshold = 0.38
y_pred = (hybrid_probs >= threshold).astype(int)
print('[*] Dual-Engine training completed successfully.')

---
## 6. Model Evaluation & CBE Regulatory Compliance

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

recall = float(recall_score(y_test, y_pred))
precision = float(precision_score(y_test, y_pred))
f1 = float(f1_score(y_test, y_pred))
f2 = float(fbeta_score(y_test, y_pred, beta=2.0))
roc_auc = float(roc_auc_score(y_test, hybrid_probs))
pr_auc = float(average_precision_score(y_test, hybrid_probs))
fpr = float(fp / (tn + fp))

print('=' * 60)
print('     CRÉDIX DUAL-ENGINE FRAUD MODEL — AUDIT REPORT      ')
print('=' * 60)
print(f'Test Applications Evaluated : {len(y_test):,}')
print(f'Total Fraud in Test Set     : {y_test.sum():,}')
print(f'Captured Fraud (TP)         : {tp:,} of {y_test.sum()} ({recall*100:.2f}% Recall)')
print(f'Missed Camouflaged (FN)     : {fn:,} cases ({fn/y_test.sum()*100:.2f}%)')
print(f'False Alarms / Audits (FP)  : {fp:,} of {tn+fp:,} ({fpr*100:.2f}% FPR)')
print(f'Precision                   : {precision*100:.2f}%')
print(f'F2-Score (Cost-Weighted)    : {f2:.4f}')
print(f'ROC-AUC Discrimination Power: {roc_auc:.4f}')
print(f'PR-AUC                      : {pr_auc:.4f}')
print('=' * 60)
print('CBE Minimum Recall (>= 85.0%): PASSED' if recall >= 0.85 else 'FAILED')
print('CBE Maximum FPR    (<=  4.0%): PASSED' if fpr <= 0.04 else 'FAILED')

---
## 7. Saving Production Artifacts

In [ ]:
out_dir = os.path.join('..', 'model', 'artifacts', 'fraud')
os.makedirs(out_dir, exist_ok=True)

joblib.dump(iso_model, os.path.join(out_dir, 'isolation_forest_v2.joblib'))
joblib.dump(gb_model,  os.path.join(out_dir, 'fraud_gradient_boost_v2.joblib'))
joblib.dump(scaler,    os.path.join(out_dir, 'scaler_v2.joblib'))

with open(os.path.join(out_dir, 'feature_names.json'), 'w') as f:
    json.dump(FEATURE_NAMES, f, indent=4)

print('All model weights and configurations serialized to:', os.path.abspath(out_dir))
for fname in os.listdir(out_dir):
    p = os.path.join(out_dir, fname)
    print(f' - {fname:<35} {os.path.getsize(p)/1024:.1f} KB')